# Scrap notebooks

This notebook is just a compilation of scrapy tests to view functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.optimize as opt
import scipy.integrate as integr

import sympy as sym

In [ ]:
%matplotlib widget

### Plotting solutions

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.set_xlabel(r'$\Delta_d$')
ax.set_ylabel(r'$\Delta_p^{\text{min}}$')

N_value = 10000
s_value = 2000
M_value = 3

Delta_p_max_minus_min_value = 15
alpha_value = 2
beta_value = 1
gamma_value = 4
delta_value = beta_value * gamma_value / (beta_value + gamma_value)
phi_value = 0.3
f_value = 1
E_value = 1

eps = 0.5
dim = 200
alpha = 0.4

Delta_d_values = np.linspace(-50, 50, dim)
Delta_p_min_values = np.linspace(-50, 50, dim)
Delta_d_values, Delta_p_min_values = np.meshgrid(Delta_d_values, Delta_p_min_values)
F_func = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = Delta_p_min_values)
Delta_p_full_values = F_func.ppf(M_value / (M_value + 1))

zone = np.zeros((dim, dim))


SC_a_values = zone.copy()
SC_a_values[-2 * delta_value * N_value / (phi_value * s_value) - Delta_d_values <=0] = np.nan
SC_a_values[Delta_p_min_values - E_value - f_value - Delta_d_values - delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_a_values, label="a", alpha=alpha, color="blue")


SC_b_values = zone.copy()
SC_b_values[M_value * (f_value + E_value - Delta_p_full_values) - Delta_d_values - 2 * delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
SC_b_values[E_value + f_value + Delta_d_values + delta_value * N_value / (phi_value * s_value) - Delta_p_full_values <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_b_values, label="b", alpha=alpha, color="orange")

SC_c_values = zone.copy()
SC_c_values[delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value < Delta_p_min_values] = np.nan
SC_c_values[delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value > Delta_p_full_values] = np.nan
SC_c_values[Delta_d_values + delta_value * N_value / (phi_value * s_value) * (2 - F_func.cdf(delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value)) >= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_c_values, label="c", alpha=alpha, color="green")

SC_defgh_values = zone.copy()
SC_defgh_values[M_value * (Delta_p_min_values - f_value - E_value) + Delta_d_values - (M_value + 1) * 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan
SC_defgh_values[Delta_d_values - 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan

ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_defgh_values, label="defgh", alpha=alpha, color="red")


SC_i_values = zone.copy()
SC_i_values[-2 * delta_value * N_value /(s_value * phi_value) > Delta_d_values] = np.nan
SC_i_values[2 * delta_value * N_value /(s_value * (1 - phi_value)) < Delta_d_values] = np.nan
SC_i_values[2 * (Delta_p_min_values - E_value - f_value) - Delta_d_values * (1 + phi_value) - 2 * delta_value * N_value / s_value <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_i_values, label="i", alpha=alpha, color="purple")


SC_j_values = zone.copy()
SC_j_values[-Delta_d_values + M_value * (f_value + E_value) < M_value * Delta_p_min_values - 2 * delta_value * N_value * (M_value + 1) / ((1 - phi_value) * s_value)] = np.nan
SC_j_values[-Delta_d_values + M_value * (f_value + E_value) > M_value * Delta_p_full_values + 2 * delta_value * N_value / (phi_value * s_value)] = np.nan

N_d_j_lim = np.maximum(0,
                       phi_value * (1 - phi_value) * (2 * N_value / (1 - phi_value) - Delta_d_values * s_value / delta_value) / (M_value + 2 + M_value * phi_value)
                      )

#N_d_j_lim[N_d_j_lim < 0] = np.nan
#N_d_j_lim[N_d_j_lim > N_value / (M_value + 1)] = np.nan

SC_j_values[2 * delta_value * (M_value + 1) / s_value *(-N_value/(1 - phi_value) + (1 + M_value * phi_value) / (phi_value * (1 - phi_value)) * N_d_j_lim)\
            + Delta_d_values\
            + M_value*(F_func.ppf(M_value * N_d_j_lim / N_value) - f_value - E_value)\
            > 0] = np.nan

SC_j_values[np.isnan(2 * delta_value * (M_value + 1) / s_value *(-N_value/(1 - phi_value) + (1 + M_value * phi_value) / (phi_value * (1 - phi_value)) * N_d_j_lim)\
            + Delta_d_values\
            + M_value*(F_func.ppf(M_value * N_d_j_lim / N_value) - f_value - E_value))\
           ] = np.nan



ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_j_values, label="j", alpha=alpha, color="brown")

# case k is to hard to plot (solution should be solved for every point)


ax.view_init(-90, 0, 0)
ax.legend()

In [ ]:
def solution_phi_fixed(N=N_value,
                       s=s_value,
                       delta=delta_value,
                       phi=phi_value,
                       f=f_value,
                       E=E_value,
                       M=M_value,
                       Delta_d=Delta_d_values,
                       F=stats.uniform(loc=1, scale = 3)):

    def SC(N_d, N_p, N_s):
        return delta / s * (N_s**2 / (1 - phi) + N_d * (N_d + N_p) / phi)\
        + (N - N_p) * (f+E)\
        + N_d * Delta_d
        + N * integr.quad(lambda x: x * F.pdf(x), F.ppf(0), F.ppf(N_p / N))

    N_ds = []
    N_ps = []
    N_ss = []
    SCs = []
    cases = []

    # check case a
    N_d = N
    N_p = 0
    N_s = 0

    print(2 * delta * N / (phi * s))
    if 2 * delta * N / (phi * s) + Delta_d < 0 and \
    delta * N / (phi * s) + f + E + Delta_d < F.ppf(0):
        case_a_accepted = True
    else:
        case_a_accepted = False

    if case_a_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("a")
        # add the payments

    return N_ds, N_ps, N_ss, SCs, cases
        
    

In [ ]:
solution_phi_fixed(N = N_value,
                   s = s_value,
                   delta = delta_value,
                   phi = phi_value,
                   f = f_value,
                   E = E_value,
                   M = M_value,
                   Delta_d = -40,
                   F = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = 20),
                  )

In [ ]:
N_sym, N_d_sym, M_sym, phi_sym, delta_sym, s_sym = sym.symbols("N, N_d, M, varphi, delta, s")

In [ ]:
d_N_d_d_N_expr = phi_sym / (1 + M_sym * phi_sym)
d_N_d_d_N_expr

In [ ]:
d_K_d_N_expr = (2 * (N_sym - N_d_sym * (M_sym + 1)) * (1 - (M_sym + 1) * d_N_d_d_N_expr)) / (1 - phi_sym)**2\
    - 2 * (M_sym + 1) * N_d_sym * d_N_d_d_N_expr / phi_sym**2

d2 = d_K_d_N_expr.simplify()
d2

In [ ]:
d_K_d_N_expr = 2 * (N_sym * phi_sym - N_d_sym * (M_sym + 1)) / (phi_sym * (1 - phi_sym) * (1 + M_sym * phi_sym))
d_K_d_N_expr

In [ ]:
(d2 - d_K_d_N_expr).simplify()

In [ ]:
dj_dN_d_expr = 2 * delta_sym * (M_sym + 1) / (1 - phi_sym) * (1 + M_sym * phi_sym) / phi_sym / s_sym
dj_dN_d_expr

In [ ]:
dj_dphi_expr = 2 * delta_sym * (M_sym + 1) / s_sym *\
    (-N_sym / (1 - phi_sym)**2 + (M_sym * (1 - phi_sym) * phi_sym - (1 + M_sym * phi_sym) * (1 - 2 * phi_sym)) / (phi_sym * (1 - phi_sym))**2 * N_d_sym)
dj_dphi_expr

In [ ]:
dN_d_d_phi_expr = -dj_dphi_expr / dj_dN_d_expr
d2 = dN_d_d_phi_expr.simplify()
d2

In [ ]:
dN_d_d_phi_expr = (phi_sym **2 * (N_sym - M_sym * N_d_sym) + N_d_sym * (1 - 2 * phi_sym))\
    / (phi_sym * (1 - phi_sym) * (M_sym * phi_sym + 1))
dN_d_d_phi_expr

In [ ]:
(dN_d_d_phi_expr - d2).simplify()

In [ ]:
dK_dphi_expr = 2 * (
    (N_sym - N_d_sym * (M_sym + 1)) * (-(M_sym + 1) * dN_d_d_phi_expr) / (1 - phi_sym)**2\
    + (N_sym - (M_sym + 1) * N_d_sym)**2 / (1 - phi_sym)**3
    -(M_sym + 1) * (N_d_sym * dN_d_d_phi_expr / phi_sym**2 - N_d_sym **2 / phi_sym**3)
)

d2 = dK_dphi_expr.simplify()
d2

In [ ]:
dK_dphi_expr = 2 * (N_d_sym * (M_sym + 1)- N_sym * phi_sym)**2 / (phi_sym **2 * (1 - phi_sym) **2 * (M_sym * phi_sym + 1))
dK_dphi_expr

In [ ]:
(dK_dphi_expr - d2).simplify()

In [ ]:
expr = (N_sym - N_d_sym * (M_sym + 1)) ** 2 * phi_sym **2 - (M_sym + 1) * N_d_sym **2 * (1 - phi_sym) **2

expr.expand()

In [ ]:
expr2 = (N_sym * phi_sym - N_d_sym * (M_sym + 1)) * (N_sym * (phi_sym))# - N_d_sym * phi_sym * (M_sym + 1))
expr2.expand()

In [ ]:
expr.expand() - expr2.expand()